# 面试问题：MCP Agent 怎样实现 OAuth Resource Indicator、audience 绑定与安全 token 缓存？

**一句话回答。** 对远程 MCP server，Agent 不能把“拿到一个 token”理解为可以调用任何工具。客户端先从受保护资源元数据发现授权服务器，再用 PKCE 和明确的 resource 参数申请 token；资源服务器必须验证 issuer、audience、scope、过期时间和调用上下文，并为下游服务另取 token，禁止 token passthrough。

本 Notebook 只以受控小数据实现数据合同、状态机和断言，不调用大模型、真实 OAuth、真实文件或外部工具。断言验证机制不代表生产性能、安全或合规结论。

**资料入口。** [MCP Authorization 规范](https://modelcontextprotocol.io/specification/2025-06-18/basic/authorization) 要求 HTTP MCP 客户端使用 Resource Indicators，服务端验证 token 是为自身签发；本例只模拟 claims，不发起真实 OAuth 流程。

In [ ]:
question = "MCP OAuth resource audience 绑定"  # 执行本行的状态、计算或校验逻辑。
assert "OAuth" in question  # 执行本行的状态、计算或校验逻辑。
assert 8 - 3 == 5  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 身份与资源分离

OAuth client、授权服务器、MCP resource server 和用户主体是不同角色。工具名称只描述能力，不是授权；真正允许某次调用的是面向指定 canonical resource 的短期 access token 加上 scope。把一个 CRM server 的 token 转发给支付或搜索服务会制造 confused deputy 风险。

In [ ]:
server = {"resource": "https://crm.example/mcp", "issuer": "https://auth.example", "scopes": {"customers.read", "tickets.write"}}  # 执行本行的状态、计算或校验逻辑。
resource_metadata = {"authorization_servers": (server["issuer"],), "resource": server["resource"]}  # 执行本行的状态、计算或校验逻辑。
assert resource_metadata["resource"] == server["resource"]  # 执行本行的状态、计算或校验逻辑。
assert len(resource_metadata["authorization_servers"]) == 1  # 执行本行的状态、计算或校验逻辑。
assert "tickets.write" in server["scopes"]  # 执行本行的状态、计算或校验逻辑。

## 2. 发现链路

规范化实现从 401 开始，读取 Protected Resource Metadata，选择授权服务器，再读取 Authorization Server Metadata；客户端不应根据模型文本或工具描述随意拼接登录地址。教学中用固定字典代表这些元数据，强调 discovery 结果也需要 HTTPS、allowlist 与缓存失效策略。

In [ ]:
def canonical_resource(uri):  # 执行本行的状态、计算或校验逻辑。
    return uri.rstrip("/")  # 执行本行的状态、计算或校验逻辑。
requested = canonical_resource("https://crm.example/mcp/")  # 执行本行的状态、计算或校验逻辑。
assert requested == server["resource"]  # 执行本行的状态、计算或校验逻辑。
assert canonical_resource(server["resource"]) == server["resource"]  # 执行本行的状态、计算或校验逻辑。
assert canonical_resource("https://search.example/mcp/") != server["resource"]  # 执行本行的状态、计算或校验逻辑。

## 3. resource 绑定

请求与换 token 时都带 canonical resource URI。缓存键至少区分 issuer、subject、resource 与 scope；只按 user id 缓存会把面向 A 的 token 错给 B。服务端也必须拒绝 audience 不包含自身 resource 的 token，而不是相信上游说它已经验过。

In [ ]:
pkce = {"client_id": "public-agent", "state": "random-state", "challenge": "challenge-abc", "resource": requested}  # 执行本行的状态、计算或校验逻辑。
def authorization_request(value):  # 执行本行的状态、计算或校验逻辑。
    return all(value[key] for key in ("client_id", "state", "challenge", "resource"))  # 执行本行的状态、计算或校验逻辑。
assert authorization_request(pkce)  # 执行本行的状态、计算或校验逻辑。
assert pkce["resource"] == server["resource"]  # 执行本行的状态、计算或校验逻辑。
assert "challenge" in pkce  # 执行本行的状态、计算或校验逻辑。

## 4. scope 与动作

audience 对应服务，scope 再收窄动作，例如只读客户资料不等于可退款。即便模型选择了存在的 tool，PDP/PEP 仍要检查动作、主体、租户与参数；高风险动作还应绑定独立审批，而不是把 scope 当作无限期的人类同意。

In [ ]:
token = {"issuer": server["issuer"], "sub": "u-7", "aud": server["resource"], "scope": {"customers.read"}, "exp": 120, "fingerprint": "tok-fp-1"}  # 执行本行的状态、计算或校验逻辑。
assert token["aud"] == requested  # 执行本行的状态、计算或校验逻辑。
assert "customers.read" in token["scope"]  # 执行本行的状态、计算或校验逻辑。
assert token["fingerprint"].startswith("tok-")  # 执行本行的状态、计算或校验逻辑。

## 5. 安全存储与刷新

access token 不应写入 agent trace、prompt、模型上下文或普通日志；token 应短期、加密保存并按 resource 分区。刷新 token 更敏感，应轮换、撤销和最小化；此 Notebook 只记录 token 指纹，不实现加密或真实认证。

In [ ]:
def validate_access(token_value, resource, required_scope, now):  # 执行本行的状态、计算或校验逻辑。
    return token_value["aud"] == resource and required_scope in token_value["scope"] and token_value["exp"] > now and token_value["issuer"] == server["issuer"]  # 执行本行的状态、计算或校验逻辑。
assert validate_access(token, server["resource"], "customers.read", 10)  # 执行本行的状态、计算或校验逻辑。
assert not validate_access(token, "https://search.example/mcp", "customers.read", 10)  # 执行本行的状态、计算或校验逻辑。
assert not validate_access(token, server["resource"], "tickets.write", 10)  # 执行本行的状态、计算或校验逻辑。

## 6. 下游调用

MCP server 若需要访问外部 API，应作为新的 OAuth client 为下游资源获取自己的 token。不要把从 Agent 收到的 bearer token 原样转发；这不仅会越过 audience 限制，也会让下游无法判断真实的授权关系与审计责任。

In [ ]:
cache = {}  # 执行本行的状态、计算或校验逻辑。
cache[(token["issuer"], token["sub"], token["aud"], tuple(sorted(token["scope"])))] = token["fingerprint"]  # 执行本行的状态、计算或校验逻辑。
assert len(cache) == 1  # 执行本行的状态、计算或校验逻辑。
assert (server["issuer"], "u-7", "https://search.example/mcp", ("customers.read",)) not in cache  # 执行本行的状态、计算或校验逻辑。
assert list(cache.values()) == ["tok-fp-1"]  # 执行本行的状态、计算或校验逻辑。

## 7. 验收与边界

测试要覆盖错误 audience、缺 scope、过期、资源 URI 规范化、缓存隔离、redirect/metadata allowlist 和日志脱敏。示例中的字典无法替代 HTTPS、PKCE、JWT 签名验证、密钥轮换、OIDC、DCR 和企业 IAM 集成。

In [ ]:
def downstream_token(inbound, downstream_resource):  # 执行本行的状态、计算或校验逻辑。
    return {"aud": downstream_resource, "issued_from": inbound["issuer"], "fingerprint": "down-fp-1"}  # 执行本行的状态、计算或校验逻辑。
downstream = downstream_token(token, "https://billing.example/api")  # 执行本行的状态、计算或校验逻辑。
assert downstream["aud"] == "https://billing.example/api"  # 执行本行的状态、计算或校验逻辑。
assert downstream["aud"] != token["aud"]  # 执行本行的状态、计算或校验逻辑。
assert downstream["fingerprint"] != token["fingerprint"]  # 执行本行的状态、计算或校验逻辑。

## 8. 面试追问

回答时还应区分教学状态机与生产系统：前者用小数据证明拒绝条件和版本绑定，后者还要覆盖并发、网络故障、机密管理、审计留存与真实依赖的集成测试。任何无法由当前证据确认的状态，都应显式返回未验证、降级或人工升级，而不是由模型补全。

In [ ]:
audit = {"sub": token["sub"], "resource": token["aud"], "scope": tuple(sorted(token["scope"])), "token": token["fingerprint"]}  # 执行本行的状态、计算或校验逻辑。
assert audit["token"] == "tok-fp-1"  # 执行本行的状态、计算或校验逻辑。
assert "access_token" not in audit  # 执行本行的状态、计算或校验逻辑。
assert audit["resource"] == server["resource"]  # 执行本行的状态、计算或校验逻辑。

## 面试总结

高质量回答应先给出模型或 Agent 的责任边界，再说明数据合同、状态转换、确定性 verifier 和失败处理，最后明确性能、权限与现实系统依赖的验证方法。不要把一次函数返回、模型文本或受控小样本断言误称为线上正确性。